# Perception Pipeline
This notebook will go through a perception pipeline, first writing a script that detects the Intersection over Union (IoU), then Non-Maximum Suppression (NMS), then detection matching, and finally calculating the precision statistics.

## Intersection over Union (IoU)
Intersection over Union, at its base, calculates how well 2 boxes overlap. The intersection area is the literal overlap area, and the union area is the total area of the two boxes. The Intersection area is divided over the union area to determine how much they overlap. We will assume that each box is always formatted as **[x1, y1, x2, y2]**

In [1]:
import numpy as np

BOXES1 = np.array([[0,0,10,10],[5,5,15,15]])
BOXES2 = np.array([[2,2,8,8],[5,5,15,15],[20,40,40,80]])

def area(box: np.ndarray) -> np.ndarray: # We will assume the boxes are always formatted as stated above.
    """ Returns the area of a box or column of boxes """

    boxes = np.atleast_2d(box)
    
    x1, y1, x2, y2 = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]

    return (abs(x2-x1) * abs(y2-y1))

def validate_boxes(boxes: np.ndarray) -> None:
    if boxes.shape[-1] != 4:
        raise ValueError(f'Boxes must have 4 coordinates. Got box shape {boxes.shape}')

    flipped = (boxes[:, 2] < boxes[: ,0]) | (boxes[:, 3] < boxes[:, 1])
    if np.any(flipped):
        bad = np.flatnonzero(flipped).tolist()

        raise ValueError(f"Box has reversed indices at {bad}")

def iou(boxes1: np.ndarray, boxes2: np.ndarray) -> np.ndarray:
    """ Returns a matrix of the IoUs for each box.
        Assumed that x1 < x2 and y1 < y2. There is a check to make sure this is true.
    """
    boxes1 = np.atleast_2d(boxes1)
    boxes2 = np.atleast_2d(boxes2) # Turn them both into 2d

    validate_boxes(boxes1)
    validate_boxes(boxes2)
    
    areas_b1 = area(boxes1)
    areas_b2 = area(boxes2)

    x1_1 = boxes1[:, None, 0]
    y1_1 = boxes1[:, None, 1]
    x2_1 = boxes1[:, None, 2]
    y2_1 = boxes1[:, None, 3]

    x1_2 = boxes2[:,0]
    y1_2 = boxes2[:,1]
    x2_2 = boxes2[:,2]
    y2_2 = boxes2[:,3]

    x_start = np.maximum(x1_1, x1_2)
    y_start = np.maximum(y1_1, y1_2)
    x_end = np.minimum(x2_1, x2_2)
    y_end = np.minimum(y2_1, y2_2)

    widths = np.maximum(0, x_end - x_start)
    heights = np.maximum(0, y_end - y_start)

    intersection_areas = widths * heights
    union_areas = (areas_b2 + areas_b1[:, None]) - intersection_areas


    iou_result = np.divide(intersection_areas, union_areas, 
                           out = np.zeros_like(intersection_areas, dtype=float), 
                           where= union_areas > 0) # 0 by default, so a divide by 0 doesn't break it
    
    return iou_result

print(iou(BOXES1, BOXES1),'\n')
print(iou(BOXES1, BOXES2),'\n')
print(iou(BOXES2[1], BOXES1[1]))

[[1.         0.14285714]
 [0.14285714 1.        ]] 

[[0.36       0.14285714 0.        ]
 [0.07086614 1.         0.        ]] 

[[1.]]


Testing with boxes1 against itself, it turns out as expected. The first box matches with itself completely, and has a small match with the next box.

## Non-Maximum Suppression (NMS)
The next step is to take the non-maximum suppression. An object detector usually predicts many overlapping boxes around the same object, each with a confidence score. **Non-maximum suppression** cleans this up by keeping only the best box for each object and discarding the redundant ones.

### Algorithm

Given a set of boxes, their confidence scores, and a preset IoU threshold (usually 0.5):

1. Sort the boxes by score, highest first.
2. Take the highest-scoring remaining box and keep it.
3. Compute the IoU between that box and every other remaining box.
4. Discard any remaining box whose IoU with the kept box is above the threshold. These are treated as duplicate detections of the same object.
5. Repeat steps 2–4 until no boxes remain.

This next example will have the following boxes and scores:
* Box1: 0, 0, 10, 10
* Box2: 1, 1, 11, 11
* Box3: 20, 20, 30, 30
* Box4: 0, 0, 9 ,9

1. 0.9
2. 0.8
3. 0.75
4. 0.6

So we should expect to keep box 1 and box 3 **(Which will be shown as indices 0 & 2)**


In [2]:
BOXES_NMS  = np.array([[0, 0, 10, 10], [1, 1, 11, 11], [20, 20, 30, 30], [0, 0, 9, 9]])
SCORES = np.array([0.9, 0.8, 0.75, 0.6])

def nms(boxes: np.ndarray, confidences: list | np.ndarray, threshold=0.5) -> np.ndarray:
    """Returns:
        numpy array of the indices of the kept boxes
    """

    if len(boxes) == 0 or len(confidences) == 0:
        return np.array([], dtype=int)

    if len(boxes) != len(confidences):
        raise ValueError("Boxes and confidences must be the same length!")

    order = np.argsort(confidences)[::-1]
    keep = []

    remaining = list(order)

    while remaining:
        curr_idx = remaining[0]
        curr_box = boxes[curr_idx]
        keep.append(curr_idx)
        
        other_idx = remaining[1:]
        other_boxes = boxes[other_idx]

        iou_scores = iou(curr_box,other_boxes)[0]

        remaining = [rem_idx for rem_idx, score in zip(other_idx, iou_scores) if score <= threshold]

    return np.array(keep, dtype=int)

remaining_boxes = nms(BOXES_NMS, SCORES)

print('Remaining boxes:', remaining_boxes.tolist())


Remaining boxes: [0, 2]


## Detection Matching
After NMS, we have a set of predicted boxes with a confidence score. The next step is to add in the ground-truth boxes.

A prediction counts as a correct detection of a ground-truth box if their IoU is at or above a threshold (commonly 0.5). There's one extra rule: **each ground-truth box can be matched at most once.** If two predictions both overlap the same object, only one of them gets credit, and the other is a duplicate.

### Algorithm
1. Compute the IoU matrix between all predictions and all ground-truth boxes.
2. Sort the predictions by confidence, highest first.
3. For each prediction, in that order:
   - Find the **unmatched** ground-truth box with the highest IoU.
   - If that IoU ≥ threshold, it's a **true positive (TP)**, and that ground-truth box is marked as matched.
   - Otherwise, it's a **false positive (FP)**.
4. Any ground-truth box left unmatched at the end is a **false negative (FN)**, meaning an object the detector missed.

All of these are processed in confidence order.


### Resulting metrics
- **Precision** - Of the boxes we predicted, how many were right? $\frac{TP}{TP+FP}$
- **Recall** - Of the real objects, how many did we find? $\frac{TP}{TP + FN}$



In [3]:
SCENE_GT_BOXES   = np.array([[0, 0, 10, 10], [20, 20, 30, 30]])
PRED_BOXES = np.array([[1, 1, 11, 11], [0, 0, 9, 9], [50, 50, 60, 60], [21, 21, 30, 30]])
PRED_CONFS = np.array([0.9, 0.8, 0.7, 0.6])

def match_detections(pred_boxes: np.ndarray, confidences: np.ndarray, gt_boxes: np.ndarray, threshold: float=0.5) -> dict:
    """Greedily match predictions to ground truth by confidence.
    Returns: 
    Dict of matches: 
        prediction+ground-truth pairs, 
        true positives,
        false positives,
        the indices of ground truths that weren't matched
        
    """
    matches = []
    n_pred, n_gt = len(pred_boxes), len(gt_boxes)
    order = np.argsort(-confidences)
    claimed_gt = set()

    tp = 0
    fp = 0

    if n_pred > 0 and n_gt > 0: # If they actually exist

        
        for idx_prediction in order: # prediction is the idx
            best_iou = float('-inf')
            best_gt = -1
            
            curr_pred = pred_boxes[idx_prediction]
            ious = iou(curr_pred, gt_boxes)[0]

            for idx_gt, iou_score in enumerate(ious):
                if iou_score > best_iou and idx_gt not in claimed_gt: # if it's the best one we've seen yet
                    best_iou = iou_score
                    best_gt = idx_gt

            if best_iou >= threshold: # If the best match is above the given threshold, add it as a match
                claimed_gt.add(best_gt)
                tp += 1
                matches.append([idx_prediction,best_gt])
                
            else: # Otherwise, we need to count it as a false positive
                fp += 1
        
        
    else: # If there are no ground truths
        fp = n_pred
            


    missed_gt = [g for g in range(n_gt) if g not in claimed_gt]

    # fn = len(missed_gt) -> Optional, if needed in the future
            
    result = {'matches': np.array(matches, dtype=int),
                  'tp': tp,
                  'fp': fp,
                  'fn_gt': missed_gt}

        


    return result

matched_detections = match_detections(PRED_BOXES, PRED_CONFS, SCENE_GT_BOXES)
matched_detections


{'matches': array([[0, 0],
        [3, 1]]),
 'tp': 2,
 'fp': 2,
 'fn_gt': []}

## Average Precision (AP)
AP removes the dependence on a single confidence cutoff by evaluating the detector at *every* cutoff at once.

1. Sort the predictions by confidence, highest first, and mark each as TP or FP (from detection matching).
2. Walk down the list. After each prediction, compute the precision and recall using only the predictions seen so far. This traces out a precision-recall curve, going from "keep only the most confident box" to "keep every box."
3. Smooth the curve so precision never increases as recall increases. At each recall level, use the *best* precision achievable at that recall or higher.
4. **AP is the area under this curve.**

A perfect detector ranks every true positive above every false positive and finds every object, which gives AP = 1.0. AP rewards two things: finding the objects, and being *more confident* about the correct boxes than the wrong ones.

**mAP** (mean Average Precision) is the AP averaged over all object classes. COCO additionally averages over IoU thresholds from 0.50 to 0.95, which rewards tighter boxes.

## F1 Score

Precision and recall pull against each other. A detector that predicts very few boxes can have high precision but low recall, while one that predicts boxes everywhere can have high recall but low precision. The F1 score combines them into a single number using the harmonic mean:

$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

The harmonic mean is low whenever *either* value is low. So a detector can't get a good F1 by excelling at one and ignoring the other.

F1 is measured at a **single operating point**: one confidence cutoff and one IoU threshold. If you change which predictions you keep, the F1 changes.



In [4]:
def precision_recall_f1(tp: int, fp: int, fn: int) -> dict:
    """From the match counts, returns:
        precision
        recall
        F1
    """


    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if precision + recall == 0:
        return {'precision': precision, 'recall': recall, 'f1': 0.0}

    f1 = (2 * precision * recall) / (precision + recall)

    result = {'precision': precision,
              'recall': recall,
              'f1': f1}

    return result

def precision_recall_curve(confidences: np.ndarray, matches: list | np.ndarray, n_gt: int) -> tuple[np.ndarray, np.ndarray]:
    """Returns: Raw precision and recall after each prediction, in confidence order."""

    is_tp = np.zeros(len(confidences), dtype=bool)
    for pred_idx, _ in matches:
        is_tp[pred_idx] = True

    order = np.argsort(-confidences)
    cum_tp = np.cumsum(is_tp[order])
    cum_fp = np.cumsum(~is_tp[order])

    precision = cum_tp / (cum_tp + cum_fp)
    recall = cum_tp / n_gt
    return precision, recall


def smooth_pr_curve(precision: np.ndarray, recall: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Pads the curve to span recall 0 to 1 and makes precision non-increasing."""
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([0.0], precision, [0.0]))
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    return precision, recall


def average_precision(confidences: np.ndarray, matches: list | np.ndarray, n_gt: int) -> float:
    """Returns: the interpolated AP under the precision-recall curve."""
    if n_gt == 0:
        return float('nan')
    if len(confidences) == 0:
        return 0.0

    precision, recall = precision_recall_curve(confidences, matches, n_gt)
    precision, recall = smooth_pr_curve(precision, recall)

    steps = np.flatnonzero(recall[1:] != recall[:-1])
    result = float(np.sum((recall[steps + 1] - recall[steps]) * precision[steps + 1]))
    return result

tp, fp, fn = matched_detections['tp'], matched_detections['fp'], len(matched_detections['fn_gt'])

metrics = precision_recall_f1(tp, fp, fn)
print(metrics)

ap = average_precision(PRED_CONFS, matched_detections['matches'], len(SCENE_GT_BOXES))

print(f"Average Precision: {ap}")

{'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666}
Average Precision: 0.75


## Full Pipeline Example
The earlier sections each tested one stage on its own small example. This section runs all of them together on a single scene: raw detector output -> NMS -> detection matching -> precision, recall, F1, and AP.

The scene has 4 ground-truth objects (car, person, dog, bike). The raw detections were built to include the problems a real detector produces:

- **Duplicate boxes** – three boxes on the car and two on the person, each slightly shifted.
- **A confident false alarm** – a box on empty background with 0.85 confidence.
- **A badly localized box** – a box near the bike, but with an IoU of only 0.24.
- **A correct but low-confidence box** – the dog is found, but with 0.55 confidence, below both false alarms.
- **A weak false alarm** – a background box with 0.40 confidence.

The scene is evaluated twice, once on the raw detections and once after NMS, to show what NMS changes.

In [5]:
# Ground truth: 4 objects in the scene
GT_BOXES = np.array([
    [ 10,  10,  60,  40],   # 0: car
    [100,  20, 120,  70],   # 1: person
    [150,  60, 190,  90],   # 2: dog
    [220,  30, 260,  70],   # 3: bike
])

# Raw detector output, before NMS
RAW_BOXES = np.array([
    [ 12,  11,  61,  41],   # 0: car, best box
    [  8,   9,  58,  39],   # 1: car, duplicate
    [ 15,  14,  65,  44],   # 2: car, duplicate (looser)
    [101,  22, 121,  72],   # 3: person, good box
    [ 98,  18, 119,  68],   # 4: person, duplicate
    [300, 100, 340, 140],   # 5: background, confident false alarm
    [235,  45, 275,  85],   # 6: bike, badly localized (IoU 0.24)
    [152,  58, 188,  92],   # 7: dog, correct but low confidence
    [ 50,  80,  70, 100],   # 8: background, weak false alarm
])
RAW_CONFS = np.array([0.95, 0.88, 0.70, 0.91, 0.60, 0.85, 0.80, 0.55, 0.40])


def evaluate(boxes, confs, gt_boxes, label):
    matched = match_detections(boxes, confs, gt_boxes)
    metrics = precision_recall_f1(matched['tp'], matched['fp'], len(matched['fn_gt']))
    ap = average_precision(confs, matched['matches'], len(gt_boxes))

    print(f"--- {label} ---")
    print(f"TP={matched['tp']}  FP={matched['fp']}  FN={len(matched['fn_gt'])}  missed GT: {matched['fn_gt']}")
    print(f"Precision={metrics['precision']:.3f}  Recall={metrics['recall']:.3f}  "
          f"F1={metrics['f1']:.3f}  AP={ap:.3f}\n")
    return matched, metrics


keep = nms(RAW_BOXES, RAW_CONFS)
print("Kept after NMS:", keep.tolist(), "\n")

evaluate(RAW_BOXES, RAW_CONFS, GT_BOXES, "Without NMS")
evaluate(RAW_BOXES[keep], RAW_CONFS[keep], GT_BOXES, "With NMS");

Kept after NMS: [0, 3, 5, 6, 7, 8] 

--- Without NMS ---
TP=3  FP=6  FN=1  missed GT: [3]
Precision=0.333  Recall=0.750  F1=0.462  AP=0.594

--- With NMS ---
TP=3  FP=3  FN=1  missed GT: [3]
Precision=0.500  Recall=0.750  F1=0.600  AP=0.650



### Results

| | TP | FP | FN | Precision | Recall | F1 | AP |
|---|---|---|---|---|---|---|---|
| Without NMS | 3 | 6 | 1 | 0.333 | 0.750 | 0.462 | 0.594 |
| With NMS | 3 | 3 | 1 | 0.500 | 0.750 | 0.600 | 0.650 |

**NMS improves precision without costing recall.** NMS removes the two car duplicates and the person duplicate. Without NMS, those three boxes still get evaluated. Each ground truth can only be matched once, so every duplicate becomes a false positive. Recall doesn't change, because NMS never removed the only box on an object.

**A badly localized box counts against you twice.** The bike box is close to the bike, but its IoU of 0.24 is below the 0.5 threshold. It counts as a false positive, and the bike counts as a missed object (false negative). Being "near" an object gives no credit.

**AP reflects confidence ranking, not just correctness.** After NMS, the predictions in confidence order are TP, TP, FP, FP, TP, FP. The two confident false alarms rank above the dog, so precision has already dropped to 0.5 by the time the dog is found. If the dog had scored higher than the false alarms, AP would go up while precision, recall, and F1 stayed exactly the same. That's the difference between the two kinds of metric: F1 only measures *which* boxes were kept, while AP also measures whether the detector was *more confident about the right ones*.